<a href="https://colab.research.google.com/github/prjhseo/study_RS/blob/main/Temporal_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-25m.zip
!unzip ml-25m.zip

# 데이터 로드

In [6]:
import csv
from datetime import datetime
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

userIds_map, itemIds_map = {}, {}
users, items, ratings, dates = [], [], [], []

with open("ml-25m/ratings.csv", "r") as f:
    reader = csv.reader(f)
    print(next(reader))
    for userId_str, itemId_str, rating_str, timestamp_str in reader:
        if userId_str not in userIds_map:
            userIds_map[userId_str] = len(userIds_map)
        if itemId_str not in itemIds_map:
            itemIds_map[itemId_str] = len(itemIds_map)

        users.append(userIds_map[userId_str])
        items.append(itemIds_map[itemId_str])
        ratings.append(float(rating_str))
        dates.append(datetime.fromtimestamp(int(timestamp_str)))

users = torch.tensor(users, device=device)
items = torch.tensor(items, device=device)
ratings = torch.tensor(ratings, device=device)

tmin = min(dates)
dates = [(t - tmin).days for t in dates] # 첫날 기준 경과일 수
dates = torch.tensor(dates, device=device)

['userId', 'movieId', 'rating', 'timestamp']


In [9]:
dates

tensor([4146, 4146, 4146,  ..., 5223, 5223, 5223])

# Training Set/ Validation Set 분할

In [10]:
n_train=int(len(ratings)*0.9)
indices=torch.randperm(len(ratings))
train_indices=indices[:n_train]
valid_indices=indices[n_train:]

users_valid=users[valid_indices]
items_valid=items[valid_indices]
ratings_valid=ratings[valid_indices]
dates_valid=dates[valid_indices]

users=users[train_indices]
items=items[train_indices]
ratings=ratings[train_indices]
dates=dates[train_indices]

# Latent Factor Model

In [ ]:
n_users = len(userIds_map)
n_items = len(itemIds_map)

mean = ratings.mean()

user_bias = torch.zeros(n_users,requires_grad=True,device=device)

item_bias = torch.zeros(n_items, requires_grad=True,device=device)

user_emb = torch.normal( 0, 0.01,size=(n_users,10),requires_grad=True,device=device)

item_emb = torch.normal(0,0.01,size=(n_items,10),requires_grad=True,device=device)

# ------------------
# Time bin
# ------------------

n_bins = 30

tmax = max( dates.max().item(),dates_valid.max().item())

date_bins = (dates * n_bins) // (tmax + 1)
date_bins_valid = (dates_valid * n_bins) // (tmax + 1)

date_bins = date_bins.long()
date_bins_valid = date_bins_valid.long()

item_bin_bias = torch.zeros((n_items, n_bins),requires_grad=True,device=device)

# ------------------
# User average date
# ------------------

tu = torch.bincount(users,weights=dates.float())

tu /= torch.bincount(users).float()

# ------------------
# Time deviation
# ------------------

dev = ((dates - tu[users]).sign()*(dates - tu[users]).abs().pow(0.4))

dev_valid = ((dates_valid - tu[users_valid]).sign()*(dates_valid - tu[users_valid]).abs().pow(0.4))

dev_bias = torch.zeros( n_users,requires_grad=True,device=device)

# ------------------
# User-day bias
# ------------------

all_users = torch.cat([users, users_valid])

all_dates = torch.cat([dates, dates_valid])

pairs = torch.stack( [all_users, all_dates],dim=1)

_, all_userdates = torch.unique( pairs,dim=0,return_inverse=True)

user_dates = all_userdates[:len(users)]
user_dates_valid = all_userdates[len(users):]

n_userdates = (all_userdates.max().item() + 1)

perday_bias = torch.zeros(n_userdates,requires_grad=True,device=device)

# ------------------
# Prediction
# ------------------

def predict( users, items, date_bins,dev,user_dates):
    h = mean.expand(len(users))
    h += user_bias[users]
    h += item_bias[items]
    h += (user_emb[users]*item_emb[items]).sum(dim=1)

    h += item_bin_bias[items,date_bins]
    h += dev_bias[users] * dev
    h += perday_bias[user_dates]

    return h

# ------------------
# Optimizer
# ------------------

optim = torch.optim.Adam([user_bias,item_bias,user_emb,item_emb,item_bin_bias,dev_bias, perday_bias],lr=0.01,weight_decay=1e-4)

mse = torch.nn.MSELoss()

# ------------------
# Training
# ------------------

for epoch in range(1000):

    h = predict( users,items,date_bins,dev,user_dates)

    cost = mse(h,ratings)

    optim.zero_grad()

    cost.backward()

    optim.step()

    with torch.no_grad():

        h_valid = predict(users_valid,items_valid,date_bins_valid,dev_valid,user_dates_valid )

        cost_valid = mse(h_valid,ratings_valid)

    if epoch % 100 == 0:

        print(
            f"epoch={epoch} "
            f"train={cost.item():.4f} "
            f"valid={cost_valid.item():.4f}"
        )